In [1]:
import pandas as pd
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import layers, callbacks,utils
from imblearn.over_sampling import SMOTE
from collections import Counter

import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [2]:
df=pd.read_csv("cleaned_dataset_taiwan_2months.csv")
df.head()

,date,sitename,county,aqi,status,so2,co,o3,pm10,pm2.5,no2,nox,no,siteid
0,2024-08-31 23:00:00,Hukou,Hsinchu County,62.0,Moderate,0.9,0.17,35.0,18.0,17.0,2.3,2.6,0.3,22
1,2024-08-31 23:00:00,Zhongming,Taichung City,50.0,Good,1.6,0.32,27.9,27.0,14.0,7.6,9.3,1.6,31
2,2024-08-31 23:00:00,Zhudong,Hsinchu County,45.0,Good,0.4,0.17,25.1,21.0,13.0,2.9,4.1,1.1,23
3,2024-08-31 23:00:00,Hsinchu,Hsinchu City,42.0,Good,0.8,0.20,30.0,19.0,10.0,4.0,4.8,0.7,24
4,2024-08-31 23:00:00,Toufen,Miaoli County,50.0,Good,1.0,0.16,33.5,18.0,14.0,1.8,3.1,1.2,25


In [3]:
features=['so2','co','o3','pm2.5','pm10','no2','nox','no']
status=['status']
X=df[features]
y=df[status]  


In [4]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
y_encoded = utils.to_categorical(y_encoded, num_classes=num_classes)

c:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [5]:
X_train,X_test,y_train,y_test=train_test_split(X,y_encoded,test_size=0.2,random_state=17)

In [6]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [7]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X_train_res)

In [8]:
model = tf.keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_res.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2), # Prevents overfitting
    layers.Dense(num_classes, activation='softmax') # Softmax for multi-class
])

c:\Users\Administrator\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

In [10]:
my_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(filepath='best_model.keras', monitor='val_accuracy', save_best_only=True)
]

In [11]:
model.fit(
X_train_res, y_train_res,
epochs=10,
batch_size=32,
validation_split=0.1,
verbose=1
)#callbacks=my_callbacks

Epoch 1/10
7758/7758 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - accuracy: 0.7804 - loss: 0.5303 - precision: 0.8133 - recall: 0.7336 - val_accuracy: 0.9215 - val_loss: 0.3018 - val_precision: 0.9426 - val_recall: 0.9044
Epoch 2/10
7758/7758 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.8223 - loss: 0.4197 - precision: 0.8429 - recall: 0.7988 - val_accuracy: 0.8987 - val_loss: 0.3761 - val_precision: 0.9288 - val_recall: 0.8652
Epoch 3/10
7758/7758 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.8354 - loss: 0.3897 - precision: 0.8504 - recall: 0.8194 - val_accuracy: 0.7359 - val_loss: 0.7074 - val_precision: 0.7556 - val_recall: 0.7153
Epoch 4/10
7758/7758 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - accuracy: 0.8430 - loss: 0.3737 - precision: 0.8554 - recall: 0.8301 - val_accuracy: 0.9733 - val_loss: 0.2334 - val_precision: 0.9847 - val_recall: 0.9602
Epoch 5/10
7758/7758 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.8494 - loss: 0.3614 - precision: 0.8598 - recall: 0.8396 - val_accuracy: 0.9937

In [12]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

#  PREDICT
y_pred = rf_model.predict(X_test)

# CALCULATE GLOBAL METRICS
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

print("--- Overall Categorical Performance (Global) ---")
print(f"Accuracy:  {accuracy:.2%}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# WHICH POLLUTANT AFFECTS AQI THE MOST
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
print("\n--- Pollutant Importance Ranking ---")
print(importances)

--- Overall Categorical Performance (Global) ---
Accuracy:  92.38%
Precision: 0.9347
Recall:    0.9238
F1-Score:  0.9286

--- Pollutant Importance Ranking ---
pm2.5    0.258080
o3       0.217668
pm10     0.135041
co       0.111226
no       0.085059
no2      0.071745
nox      0.063782
so2      0.057400
dtype: float64


In [13]:
import joblib

# 1. Save the trained Random Forest model
#joblib.dump(rf_model, 'rf_aqi_classifier.pkl')

# 2. Save the LabelEncoder (Crucial for decoding results later)
#joblib.dump(le, 'aqi_label_encoder.pkl')